# ZeroAlpha Colab Research Training

This notebook is meant to run inside the connected Google Colab runtime. It uses the production `zeroalpha.cli` training/backtest path, but runs a much larger lower-timeframe HPO matrix than is practical on the local laptop.

The default bootstrap clones the public GitHub branch into `/content/ZeroAlpha`, so it does not depend on the Colab upload widget.


In [ ]:
# Bootstrap the repo into /content/ZeroAlpha.
# If /content/ZeroAlpha already exists from a canceled run, this cell force-refreshes it
# to the requested branch so stale code cannot be reused accidentally.
from pathlib import Path
from urllib.parse import urlparse
import os
import shutil
import subprocess
import sys
import zipfile

BOOTSTRAP_MODE = "git"  # "git", "upload", or "existing"
GIT_URL = os.environ.get("ZEROALPHA_GIT_URL", "https://github.com/ea28/ZeroAlpha.git")
GIT_BRANCH = os.environ.get("ZEROALPHA_GIT_BRANCH", "codex/colab-research-training")
GIT_DEPTH = os.environ.get("ZEROALPHA_GIT_DEPTH", "1")
FORCE_FRESH_CLONE = os.environ.get("ZEROALPHA_FORCE_FRESH_CLONE", "0") == "1"
IN_COLAB_CONTENT = Path("/content").exists()
PROJECT_ROOT = Path("/content/ZeroAlpha") if IN_COLAB_CONTENT else Path.cwd()

def run_command(command: list[str], *, check: bool = False) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(safe_git_url(part) for part in command))
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit_code={completed.returncode}: {' '.join(command)}")
    return completed

def find_repo_root(root: Path) -> Path | None:
    for candidate in [root, *root.glob("*")]:
        if (candidate / "src" / "zeroalpha").exists():
            return candidate
    return None

def colab_secret(name: str) -> str:
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def authenticated_git_url(url: str) -> str:
    token = colab_secret("GITHUB_TOKEN") or colab_secret("GH_TOKEN")
    parsed = urlparse(url)
    if token and parsed.scheme == "https" and parsed.netloc == "github.com":
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url

def safe_git_url(url: str) -> str:
    parsed = urlparse(url)
    if parsed.username or parsed.password:
        return url.replace(f"{parsed.username}:{parsed.password}@", "***@")
    return url

def verify_branch_exists(clone_url: str) -> None:
    refs = run_command(["git", "ls-remote", "--heads", clone_url], check=False)
    if refs.returncode != 0:
        raise RuntimeError(
            "Could not read GitHub refs. If the repo is private, add a Colab secret named "
            "GITHUB_TOKEN with repo read access, or make sure the repo is public."
        )
    branch_ref = f"refs/heads/{GIT_BRANCH}"
    if branch_ref not in refs.stdout:
        raise RuntimeError(
            f"Branch {GIT_BRANCH!r} was not found on {GIT_URL}. Available heads were printed above."
        )

def clone_repo(clone_url: str) -> None:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    command = [
        "git",
        "clone",
        "--depth",
        str(GIT_DEPTH),
        "--branch",
        GIT_BRANCH,
        "--single-branch",
        clone_url,
        str(PROJECT_ROOT),
    ]
    completed = run_command(command, check=False)
    if completed.returncode != 0:
        raise RuntimeError("Git clone failed. The git stdout/stderr above is the source of truth.")

def refresh_existing_git_repo(clone_url: str) -> bool:
    if not (PROJECT_ROOT / ".git").exists():
        return False
    run_command(["git", "-C", str(PROJECT_ROOT), "remote", "set-url", "origin", clone_url], check=True)
    run_command(["git", "-C", str(PROJECT_ROOT), "fetch", "--depth", str(GIT_DEPTH), "origin", GIT_BRANCH], check=True)
    run_command(["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_BRANCH, "FETCH_HEAD"], check=True)
    run_command(["git", "-C", str(PROJECT_ROOT), "reset", "--hard", "FETCH_HEAD"], check=True)
    return True

if BOOTSTRAP_MODE == "git" and IN_COLAB_CONTENT:
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    clone_url = authenticated_git_url(GIT_URL)
    verify_branch_exists(clone_url)
    if FORCE_FRESH_CLONE or not refresh_existing_git_repo(clone_url):
        clone_repo(clone_url)
elif BOOTSTRAP_MODE == "git" and (PROJECT_ROOT / "src" / "zeroalpha").exists():
    print("Using existing local checkout; git refresh is only automatic under /content.")
elif not (PROJECT_ROOT / "src" / "zeroalpha").exists():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if BOOTSTRAP_MODE == "upload":
        try:
            from google.colab import files
        except Exception as exc:
            raise RuntimeError("upload bootstrap requires a Colab notebook runtime") from exc
        print("Upload zeroalpha_colab_bundle.zip from the local repo root.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if not zip_names:
            raise RuntimeError("no .zip file uploaded")
        upload_path = Path(zip_names[0])
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(upload_path) as archive:
            archive.extractall(PROJECT_ROOT)
        discovered = find_repo_root(PROJECT_ROOT)
        if discovered is not None and discovered != PROJECT_ROOT:
            temp_root = PROJECT_ROOT.with_name("ZeroAlpha_extracted")
            if temp_root.exists():
                shutil.rmtree(temp_root)
            discovered.rename(temp_root)
            shutil.rmtree(PROJECT_ROOT)
            temp_root.rename(PROJECT_ROOT)
    elif BOOTSTRAP_MODE != "existing":
        raise ValueError("BOOTSTRAP_MODE must be upload, git, or existing")

if not (PROJECT_ROOT / "src" / "zeroalpha").exists():
    raise RuntimeError(f"ZeroAlpha repo not found at {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
commit = run_command(["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"], check=False)
print("PROJECT_ROOT", PROJECT_ROOT)
print("GIT_BRANCH", GIT_BRANCH)
print("PROJECT_COMMIT", commit.stdout.strip() if commit.returncode == 0 else "unknown")
print("Python", sys.version)


In [ ]:
# Dependency check for the Colab H100 runtime.
# Do not pin or reinstall NumPy/Pandas/SciPy here. Colab GPU images have many preinstalled
# binary packages tied to the runtime ABI, and changing NumPy inside the notebook can break pandas.
# We only install missing trading-model libraries, using --no-deps so pip does not rewrite the core stack.
import importlib.util
import subprocess
import sys

OPTIONAL_MODEL_PACKAGES = {
    "lightgbm": "lightgbm>=4.5,<4.7",
    "xgboost": "xgboost>=2.1,<3.1",
    "catboost": "catboost>=1.2.7,<1.3",
}
missing = [spec for module, spec in OPTIONAL_MODEL_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing model packages without dependencies:", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps", *missing],
        check=True,
    )
else:
    print("Model packages already available; no pip install needed.")

import numpy as np
import pandas as pd
import scipy
import sklearn
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)

# Optional foundation models. They can be slow and are not required for the broad tree-model HPO sweep.
# If you enable them, install in a fresh runtime before the long matrix and run a smoke test.
# %pip install -q --upgrade --no-deps tabpfn tabicl


In [ ]:
# Runtime / accelerator diagnostics and H100-oriented model settings.
import json
import os
import platform
import subprocess

os.environ.setdefault("ZEROALPHA_USE_GPU", "1")
os.environ.setdefault("ZEROALPHA_XGBOOST_DEVICE", "cuda")
os.environ.setdefault("ZEROALPHA_GPU_DEVICES", "0")
os.environ.setdefault("ZEROALPHA_MODEL_N_JOBS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

print("platform", platform.platform())
print("cwd", Path.cwd())
print("ZEROALPHA_USE_GPU", os.environ["ZEROALPHA_USE_GPU"])
try:
    print(subprocess.check_output(["nvidia-smi"], text=True)[-4000:])
except Exception as exc:
    print("nvidia-smi unavailable", exc)
try:
    import torch
    print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "device_count", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("cuda_device", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch unavailable", exc)

required_smoke_models = "logistic,histgb,extratrees,lightgbm,xgboost"
required_smoke = subprocess.run(
    [sys.executable, "-m", "zeroalpha.cli", "model", "smoke", "--models", required_smoke_models, "--timeout-seconds", "180"],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print(required_smoke.stdout[-6000:])
if required_smoke.returncode:
    print(required_smoke.stderr[-6000:])
    raise SystemExit(required_smoke.returncode)

catboost_smoke = subprocess.run(
    [sys.executable, "-m", "zeroalpha.cli", "model", "smoke", "--models", "catboost", "--timeout-seconds", "180"],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)
print(catboost_smoke.stdout[-4000:])
if catboost_smoke.returncode:
    print("CatBoost GPU smoke failed; full matrix will continue without catboost.")
    print(catboost_smoke.stderr[-4000:])
    os.environ["ZEROALPHA_INCLUDE_CATBOOST"] = "0"
else:
    os.environ["ZEROALPHA_INCLUDE_CATBOOST"] = "1"
print("ZEROALPHA_INCLUDE_CATBOOST", os.environ["ZEROALPHA_INCLUDE_CATBOOST"])


In [ ]:
# Configure local Colab storage for data cache and artifacts.
# This avoids Google Drive by default. Files in /content disappear when the runtime is recycled,
# so download the artifact archive from the final cell after important runs.
from datetime import datetime, timezone

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE and Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ZeroAlphaColab")
else:
    STORAGE_ROOT = PROJECT_ROOT

RUN_ID = os.environ.get("ZEROALPHA_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
CACHE_DIR = STORAGE_ROOT / "data" / "raw" / "binance"
ARTIFACT_DIR = STORAGE_ROOT / "artifacts" / "colab" / RUN_ID
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("USE_GOOGLE_DRIVE", USE_GOOGLE_DRIVE)
print("STORAGE_ROOT", STORAGE_ROOT)
print("CACHE_DIR", CACHE_DIR)
print("ARTIFACT_DIR", ARTIFACT_DIR)


In [ ]:
# Build the H100 lower-timeframe HPO matrix.
from zeroalpha.models.colab_research import (
    DEFAULT_MODELS,
    FAST_MODELS,
    H100_MODELS,
    FOUNDATION_MODELS,
    build_signal_audit_command,
    experiment_artifact_path,
    foundation_model_stack_from_smoke,
    make_foundation_kronos_experiments,
    prefetch_binance_cache,
    make_cost_stress_experiments,
    build_experiment_matrix,
    run_experiments,
    summarize_artifacts,
    write_experiment_manifest,
    write_research_report,
    write_summary_csv,
)

# H100 profile: includes GPU-capable XGBoost and CatBoost when CatBoost GPU passes smoke.
MODEL_STACK = H100_MODELS if os.environ.get("ZEROALPHA_INCLUDE_CATBOOST", "1") == "1" else FAST_MODELS
EXPERIMENTS = build_experiment_matrix(
    include_15m=True,
    include_5m=True,
    include_1m=True,
    models=MODEL_STACK,
    years_15m=7,
    years_5m=5,
    years_1m=3,
)
manifest_path = ARTIFACT_DIR / "experiment_manifest.json"
write_experiment_manifest(EXPERIMENTS, manifest_path)
print("models", MODEL_STACK)
print("experiments", len(EXPERIMENTS))
print("manifest", manifest_path)
for experiment in EXPERIMENTS[:12]:
    print(experiment.name)


In [ ]:
# Prefetch Binance archive data once before the pilot/full matrix.
# This warms /content/ZeroAlpha/data/raw/binance so each experiment reuses verified local zip files.
PREFETCH_BINANCE_CACHE = True
if PREFETCH_BINANCE_CACHE:
    cache_manifest = ARTIFACT_DIR / "cache_manifest.json"
    cache_report = prefetch_binance_cache(
        EXPERIMENTS,
        cache_dir=CACHE_DIR,
        manifest_path=cache_manifest,
    )
    print("cache manifest", cache_manifest)
    print("cache summary", cache_report["summary"])
else:
    print("Binance cache prefetch skipped; experiments will download on demand.")


In [ ]:
# Pilot: run a representative 15m subset first to verify data, speed, logs, and artifact writing.
# The pilot starts clean by default so failed/canceled runs from an older checkout are not reused.
PILOT_RESUME = False
PILOT = [
    experiment for experiment in EXPERIMENTS
    if experiment.interval == "15m"
    and "champion_types" in experiment.name
    and experiment.selection_score == "expected_utility"
][:6]
if not PILOT:
    raise RuntimeError("Pilot selection is empty; check the experiment matrix filters.")

pilot_command = build_signal_audit_command(
    PILOT[0],
    artifact_dir=ARTIFACT_DIR,
    cache_dir=CACHE_DIR,
    python_executable=sys.executable,
)
assert "--allow-data-gaps" in pilot_command, "pilot command must tolerate documented Binance archive gaps"
print("pilot experiments", len(PILOT))
print("first pilot command", " ".join(pilot_command))

if not PILOT_RESUME:
    for experiment in PILOT:
        artifact_path = experiment_artifact_path(ARTIFACT_DIR, experiment)
        log_path = ARTIFACT_DIR / "logs" / f"{experiment.name}.log"
        artifact_path.unlink(missing_ok=True)
        log_path.unlink(missing_ok=True)
    for stale_summary in ["pilot_summary.csv", "pilot_report.json"]:
        (ARTIFACT_DIR / stale_summary).unlink(missing_ok=True)

pilot_results = run_experiments(
    PILOT,
    artifact_dir=ARTIFACT_DIR,
    cache_dir=CACHE_DIR,
    python_executable=sys.executable,
    resume=PILOT_RESUME,
    stream_output=False,
)
summary_csv = ARTIFACT_DIR / "pilot_summary.csv"
write_summary_csv(pilot_results, summary_csv)
write_research_report(pilot_results, ARTIFACT_DIR / "pilot_report.json")
print("pilot summary", summary_csv)
pilot_results[:5]


In [ ]:
# Full matrix: 15m 7y, 5m 5y, and 1m 3y, all with fold-local HPO.
# This is intentionally long. Rerun this cell to resume; completed JSON artifacts are skipped.
RUN_FULL_MATRIX = True
if RUN_FULL_MATRIX:
    full_results = run_experiments(
        EXPERIMENTS,
        artifact_dir=ARTIFACT_DIR,
        cache_dir=CACHE_DIR,
        python_executable=sys.executable,
        resume=True,
        timeout_seconds=None,
        stream_output=False,
    )
else:
    full_results = summarize_artifacts(ARTIFACT_DIR)

summary_csv = ARTIFACT_DIR / "summary.csv"
report_json = ARTIFACT_DIR / "research_report.json"
write_summary_csv(full_results, summary_csv)
write_research_report(full_results, report_json)
print("wrote", summary_csv)
print("wrote", report_json)
full_results[:20]


In [ ]:
# Rank and inspect champions.
from dataclasses import asdict
import json
import pandas as pd

results = summarize_artifacts(ARTIFACT_DIR)
df = pd.DataFrame([asdict(r) for r in results])
display_cols = [
    "name", "trades", "trades_per_day", "raw_candidates_per_day", "net_pnl",
    "sharpe", "profit_factor", "max_drawdown", "hit_rate", "samples", "status", "artifact", "log_path",
]
if not df.empty:
    ranked = df.sort_values(["sharpe", "net_pnl", "trades"], ascending=[False, False, False])
    display(ranked[display_cols].head(25))
    positive = df[(df["status"] == "ok") & (df["sharpe"] > 0) & (df["net_pnl"] > 0)]
    print("ok", int((df["status"] == "ok").sum()), "of", len(df))
    print("positive sharpe+pnl", len(positive))
    for threshold in [0.5, 1, 2, 4]:
        subset = positive[positive["trades_per_day"] >= threshold]
        print(f"positive tpd >= {threshold}:", len(subset))
        if not subset.empty:
            display(subset.sort_values(["sharpe", "net_pnl"], ascending=[False, False])[display_cols].head(10))
    report_path = ARTIFACT_DIR / "research_report.json"
    if report_path.exists():
        report = json.loads(report_path.read_text())
        print("report keys", sorted(report.keys()))
else:
    print("no result artifacts yet")


In [ ]:
# Final champion stage: foundation models + Kronos on only the best broad-sweep configs.
# This is the expensive final-selection pass, not the broad search.
import importlib.util

FOUNDATION_TOP_N = 12
FOUNDATION_RESUME = True
PREFER_OFFICIAL_KRONOS = os.environ.get("ZEROALPHA_PREFER_OFFICIAL_KRONOS", "0") == "1"
FOUNDATION_PACKAGE_SPECS = {
    "tabicl": "tabicl>=2.1",
    "tabpfn": "tabpfn>=7.1.1",
}
missing_foundation = [spec for module, spec in FOUNDATION_PACKAGE_SPECS.items() if importlib.util.find_spec(module) is None]
if missing_foundation:
    print("Installing missing foundation packages without dependencies:", missing_foundation)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps", *missing_foundation], check=True)
else:
    print("Foundation packages already available; no pip install needed.")

foundation_smoke = subprocess.run(
    [sys.executable, "-m", "zeroalpha.cli", "model", "smoke", "--models", "tabicl,tabpfn", "--timeout-seconds", "240"],
    text=True,
    capture_output=True,
    env={**os.environ, "TABPFN_DISABLE_TELEMETRY": "1"},
)
print(foundation_smoke.stdout[-6000:])
if foundation_smoke.returncode:
    print("Some foundation smoke tests failed; failed models will be excluded if identifiable.")
    print(foundation_smoke.stderr[-4000:])

final_model_stack = foundation_model_stack_from_smoke(
    foundation_smoke.stdout,
    include_catboost=os.environ.get("ZEROALPHA_INCLUDE_CATBOOST", "1") == "1",
)
if "tabicl" not in final_model_stack and "tabpfn" not in final_model_stack:
    print("No TabICL/TabPFN model passed smoke; champion stage will use GPU tree models with Kronos only.")

kronos_mode = "proxy"
kronos_status = subprocess.run(
    [sys.executable, "-m", "zeroalpha.cli", "model", "kronos-status"],
    text=True,
    capture_output=True,
)
print(kronos_status.stdout[-4000:])
if PREFER_OFFICIAL_KRONOS and kronos_status.returncode == 0:
    try:
        status_payload = json.loads(kronos_status.stdout)
        if status_payload.get("official", {}).get("available"):
            kronos_mode = "official"
    except Exception:
        pass
print("foundation models", final_model_stack)
print("kronos mode", kronos_mode)

base_results = summarize_artifacts(ARTIFACT_DIR)
foundation_experiments = make_foundation_kronos_experiments(
    base_results,
    EXPERIMENTS,
    top_n=FOUNDATION_TOP_N,
    models=final_model_stack,
    kronos_mode=kronos_mode,
)
foundation_manifest = ARTIFACT_DIR / "foundation_manifest.json"
write_experiment_manifest(foundation_experiments, foundation_manifest)
print("foundation experiments", len(foundation_experiments))
print("foundation manifest", foundation_manifest)
for experiment in foundation_experiments[:12]:
    print(experiment.name)

foundation_results = run_experiments(
    foundation_experiments,
    artifact_dir=ARTIFACT_DIR,
    cache_dir=CACHE_DIR,
    python_executable=sys.executable,
    resume=FOUNDATION_RESUME,
    timeout_seconds=None,
    stream_output=False,
)
foundation_csv = ARTIFACT_DIR / "foundation_summary.csv"
foundation_report = ARTIFACT_DIR / "foundation_report.json"
write_summary_csv(foundation_results, foundation_csv)
write_research_report(foundation_results, foundation_report)
combined_report = ARTIFACT_DIR / "research_report_with_foundation.json"
write_research_report(summarize_artifacts(ARTIFACT_DIR), combined_report)
print("wrote", foundation_csv)
print("wrote", foundation_report)
print("wrote", combined_report)
foundation_results[:20]


In [ ]:
# Second-stage cost stress for the best strict champions.
# This reruns only the top positive candidates with harsher spread/slippage/fee assumptions.
# A candidate should survive this before it is treated as a serious champion.
if df.empty:
    print("No base results yet; run the pilot or full matrix first.")
else:
    stress_experiments = make_cost_stress_experiments(results, EXPERIMENTS, top_n=12)
    print("stress experiments", len(stress_experiments))
    if stress_experiments:
        stress_results = run_experiments(
            stress_experiments,
            artifact_dir=ARTIFACT_DIR,
            cache_dir=CACHE_DIR,
            python_executable=sys.executable,
            resume=True,
            timeout_seconds=None,
            stream_output=False,
        )
        stress_csv = ARTIFACT_DIR / "stress_summary.csv"
        stress_report = ARTIFACT_DIR / "stress_report.json"
        write_summary_csv(stress_results, stress_csv)
        write_research_report(stress_results, stress_report)
        print("wrote", stress_csv)
        print("wrote", stress_report)
        stress_df = pd.DataFrame([asdict(r) for r in stress_results])
        display(stress_df.sort_values(["sharpe", "net_pnl"], ascending=[False, False])[display_cols].head(25))
    else:
        print("No positive base champions available for stress testing yet.")


In [ ]:
# Optional IBKR note.
# Colab usually cannot reach a local TWS/Gateway on your Mac unless you expose it on a reachable host/VPN/tunnel.
# For now, Binance archive is the deep-history source and IBKR quotes remain best for local execution-cost calibration.
IBKR_HOST = os.environ.get("ZEROALPHA_IBKR_HOST", "")
if IBKR_HOST:
    print("IBKR_HOST configured:", IBKR_HOST)
else:
    print("IBKR not configured in Colab. Use local IBKR quote recording for spread/slippage calibration.")


In [ ]:
# Bundle artifacts for download.
CREATE_CACHE_ARCHIVE = False
archive_path = ARTIFACT_DIR.with_suffix(".tar.gz")
subprocess.run(["tar", "-czf", str(archive_path), "-C", str(ARTIFACT_DIR.parent), ARTIFACT_DIR.name], check=True)
print("artifact archive", archive_path)
if CREATE_CACHE_ARCHIVE:
    cache_archive_path = CACHE_DIR.with_suffix(".tar.gz")
    subprocess.run(["tar", "-czf", str(cache_archive_path), "-C", str(CACHE_DIR.parent), CACHE_DIR.name], check=True)
    print("cache archive", cache_archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as exc:
    print("download skipped", exc)
